# Companion — Setup & Test

Run top to bottom. Each stage confirms one subsystem before the next stage uses it — so failures are localised instantly.

Select the **companion_env** kernel in the top-right, or activate it in a terminal:
```bash
source companion_env/bin/activate
```

## Stage 0 — Install

**Run `scripts/setup.sh` in a terminal first** (it needs sudo — Jupyter can't prompt). It installs system packages, CUDA deps, the CH341 serial driver for the ESP32 screen, builds llama.cpp with CUDA, and sets up the venv.

In [ ]:
import sys, os
os.environ['PATH'] = os.path.dirname(sys.executable) + ':' + os.environ.get('PATH', '')
print('Python:', sys.executable)

# If you haven't run `bash scripts/setup.sh` in a terminal yet, stop now and do that.
# (It needs sudo access for system packages + the CH341 kernel driver.)
!pip install -r requirements.txt

### Download every model

LLM (Gemma 4 E2B + E4B), VLM (Moondream), tool router, STT (Parakeet), TTS (Kokoro + Piper), vision (YuNet + HSEmotion), EOU, speaker ID. Idempotent — re-run any time.

In [ ]:
!python3 scripts/download_models.py

### Flash the ESP32 face firmware (once)

With the Diymore 2.8" screen plugged into a USB-C port on the Jetson.

In [ ]:
!bash scripts/flash_firmware.sh

## Stage 1 — Environment sanity

Confirms every library is importable, every model is on disk, and every device is detected.

In [ ]:
!python3 scripts/verify.py

## Stage 2 — Subsystems (each opens the debug GUI or runs a focused CLI test)

Work through these one at a time. If anything fails, you know exactly which subsystem is broken before moving on.

### 2.1 Audio + DOA — mic direction visualisation

Opens a live window showing the mic waveform, VAD probability, RMS meter, and the ReSpeaker direction-of-arrival compass. Speak from different positions — the compass arrow should follow your voice.

In [ ]:
!python3 -m tests.debug_gui

(Select the **Audio** tab. Click Start.)

### 2.2 Speaker voices — headless quick test

Plays the same sentence with the current voice. Edit `tts.voice` in `config.yaml` and re-run to compare.

In [ ]:
!python3 -m tests.cli tts "Hello. I'm your companion. How does this voice sound?"

### 2.3 Vision + emotion

10-second emotion benchmark. Sit in front of the camera — you should see your current label, valence, arousal, and FPS.

In [ ]:
!python3 -m tests.cli vision --seconds 10

### 2.4 Scene understanding (VLM)

Hold an object in front of the camera and ask.

In [ ]:
!python3 -m tests.cli vlm "What am I holding?"

### 2.5 LLM chat — one-shot latency check

Prints tokens-per-second so you can compare Gemma 4 E2B vs E4B (swap via `config.yaml: llm.model`).

In [ ]:
!python3 -m tests.cli llm "Introduce yourself in two short sentences."

### 2.6 Face display — drive the screen

Drives the ESP32 screen if `display.backend: esp32_serial`, otherwise opens a pygame window on HDMI. Cycle through presets.

In [ ]:
!python3 -m tests.cli face happy --seconds 3
!python3 -m tests.cli face sad --seconds 3
!python3 -m tests.cli face surprised --seconds 3
!python3 -m tests.cli face angry --seconds 3
!python3 -m tests.cli face neutral --seconds 3

### 2.7 Memory

Stores and retrieves via Mem0 + ChromaDB, scoped by speaker.

In [ ]:
!python3 -m tests.cli mem add "User likes chai and reads on Tuesday evenings." --speaker Yogee
!python3 -m tests.cli mem search "what does user like" --speaker Yogee

### 2.8 Tool routing (FunctionGemma)

Small 270 M sidecar model parses a user utterance into a tool call.

In [ ]:
!python3 -m tests.cli tools "Set a timer for five minutes"
!python3 -m tests.cli tools "What time is it?"

### 2.9 Head motors — face tracking

Final subsystem test. Opens a popup window with the live camera feed; the YOLO26n-pose face detector tags your face and the head motors physically follow. `sim=False` drives the real servos through your calibrated `pan_limits_deg` / `tilt_limits_deg` (§8 of the motor bring-up).

**Prereqs:** head mounted, motors powered (7.4–12 V), calibration saved to `config.yaml`.

**Safety:** keep a hand near the USB-power connector. Press `q` in the popup or close the window to stop early; `Ctrl-C` also works. On exit the tracker commands `(0°, 0°)` and waits for the head to settle before torque disables.

Full bring-up (ID assignment, single-turn fix, calibration) is in [companion/motor/head_motor_quickstart.ipynb](companion/motor/head_motor_quickstart.ipynb).

In [ ]:
import importlib, sys
for _m in ('companion.vision.face_track_demo',):
    if _m in sys.modules: importlib.reload(sys.modules[_m])
from companion.vision.face_track_demo import run

run(
    sim=False,         # REAL motors — head will physically move
    kp=0.3,
    deadband=4.0,
    hfov_deg=62.0,
    vfov_deg=37.0,
    update_hz=15.0,
    duration_s=20.0,
    display='window',
)

## Stage 3 — Combined tests

Everything together. Subsystems must already pass above. The `all` target runs the sanity checks sequentially and prints a final summary.

In [ ]:
!python3 -m tests.cli all

## Stage 4 — Launch the full app

Main window comes up, face animates on the touchscreen (or HDMI fallback), speak with SPACE held down.

Run this in a terminal — notebooks don't host Qt windows well:
```bash
source companion_env/bin/activate
python3 main.py
```